In [72]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot

# Lab 9 - Multi-layer Perceptron Forward Pass & Backpropagation

## Part I
For this exercise you will implement a simple 2-layer perceptron with the forward pass and the backpropagation to learn the weights

For the first part you'll build and train a 2-layer neural network that predicts the prices of houses, using the usual Boston housing dataset.

In [73]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
boston = pd.read_table("housing.txt", names=housing_names, sep="\s+")

As usual, consider the MEDV as your target variable. 
* Split the data into training, validation and testing (70,15,15)%
* Experiment with different number of neurons per layer for your network, using the validation set

In [74]:
X = boston.values[:,:-1]
y = boston.values[:,-1]

In [75]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size= 0.7)
X_test, X_val, y_test, y_val = train_test_split(X_aux, y_aux, train_size= 0.5)


In [ ]:
class ActivationFunction:
    
    def __init__(self, function, derivative = None):
        self.function = function
        self.derivative = derivative
        
def sigmoid(a): return 1/(1 + np.exp(-a))
def sigmoid_prime(a): return sigmoid(a) * (1 - sigmoid(a))

sigmoid = ActivationFunction(sigmoid, sigmoid_prime)

def softmax(a):
    exp_a = np.exp(a)
    return exp_a / exp_a.sum(axis=1, keepdims=True)

def identity(a): return a

In [ ]:
class MultilayerPerceptron:

    def __init__(self, hidden_activation: ActivationFunction, output_activation, sizes: list):
        """A lista "sizes" deve conter todos as dimensões das camadas da nossa rede, desde a dimensão de entrada
        até a de saída. Tanto os pesos, quanto os biases estarão na matriz de parâmetros."""

        self.num_layers = len(sizes) - 1
        self.sizes = sizes
        self.hidden_activation = hidden_activation.function
        self.hidden_activation_prime = hidden_activation.derivative
        self.output_activation = output_activation
        self.parameters = [np.random.randn(x + 1,y) for x,y in zip(sizes[:-1], sizes[1:])]

    @staticmethod
    def _add_ones(X):
        """Essa função vai nos permitir adicionar uma coluna de uns aos nossos dados, para conseguirmos 
        multiplicar pelos biases."""

        return np.column_stack((np.ones(X.shape[0]), X))
    
    def forward_pass(self, X):
        z = self._add_ones(X)
        activations = [z]

        for i in range(self.num_layers):
            a = z @ w_hidden
            pre_activations.append(a)
            z = self.hidden_activation(a) 
            z = self._add_ones(z)
            data.append(z)


        w_output = self.parameters[-1]
        a = z@w_output
        pre_activations.append(a)
        y = self.output_activation(a)

        return y, data, pre_activations

    @staticmethod
    def _output_derivative(y_prev, target):
        return y_prev - target
    
    def backpropagation(self, X, y):
        n = X.shape[0]
        delta_w = [np.zeros_like(w) for w in self.parameters]

        y_prev, data, pre_activations = self.forward_pass(X)

        if y_prev.ndim == 2:
            y = y.reshape(-1, y_prev.shape[1])

        delta = self._output_derivative(y_prev, y)  
        delta_w[-1] = data[-1].T @ delta / n             

        for l in range(2, self.num_layers + 1):
            delta = (delta @ self.parameters[-l + 1][1:].T) * self.hidden_activation_derivated(pre_activations[-l])
            delta_w[-l] = data[-l].T @ delta

        return delta_w
    
    def fit(self, X_train, y_train, batch_size, lr, epochs=1):
        n_samples = X_train.shape[0]
        loss_history = []

        for epoch in range(epochs):

            # epoch_loss = 0.0
            n_batches  = 0

            for start in range(0, n_samples, batch_size):
                X_batch = X_train[start : start + batch_size]
                y_batch = y_train[start : start + batch_size]

                delta_w = self.backpropagation(X_batch, y_batch)

                for i, dw in enumerate(delta_w):
                    self.parameters[i] -= lr * dw

                y_pred, _, _ = self.forward_pass(X_batch)
                # epoch_loss += self._compute_loss(y_pred, y_batch)
                n_batches  += 1

            # epoch_loss /= n_batches
            # loss_history.append(epoch_loss)

        return loss_history
        
                
                
        
            
        
    



In [ ]:
class MultilayerPerceptron:

    def __init__(self, hidden_activation: ActivationFunction, output_activation, sizes: list):
        """A lista "sizes" deve conter todos as dimensões das camadas da nossa rede, desde a dimensão de entrada
        até a de saída. Tanto os pesos, quanto os biases estarão na matriz de parâmetros."""

        self.num_layers = len(sizes)
        self.sizes = sizes
        self.hidden_activation = hidden_activation.function
        self.hidden_activation_prime = hidden_activation.derivative
        self.output_activation = output_activation
        self.parameters = [np.random.randn(x + 1,y) for x,y in zip(sizes[:-1], sizes[1:])]

    @staticmethod
    def _add_ones(X):
        """Essa função vai nos permitir adicionar uma coluna de uns aos nossos dados, para conseguirmos 
        multiplicar pelos biases."""

        return np.column_stack((np.ones(X.shape[0]), X))
    
    def forward_pass(self, X):
        z = self._add_ones(X)
        data = [z]
        pre_activations = []

        for w_hidden in self.parameters[:-1]:
            a = z @ w_hidden
            pre_activations.append(a)
            z = self.hidden_activation(a) 
            z = self._add_ones(z)
            data.append(z)


        w_output = self.parameters[-1]
        a = z@w_output
        pre_activations.append(a)
        y = self.output_activation(a)

        return y, data, pre_activations

    @staticmethod
    def _output_derivative(y_prev, target):
        return y_prev - target
    
    def backpropagation(self, X, y):
        n = X.shape[0]
        delta_w = [np.zeros_like(w) for w in self.parameters]

        y_prev, data, pre_activations = self.forward_pass(X)

        if y_prev.ndim == 2:
            y = y.reshape(-1, y_prev.shape[1])

        delta = self._output_derivative(y_prev, y)  
        delta_w[-1] = data[-1].T @ delta / n             

        for l in range(2, self.num_layers):
            delta = (delta @ self.parameters[-l + 1][1:].T) * self.hidden_activation_derivated(pre_activations[-l])
            delta_w[-l] = data[-l].T @ delta

        return delta_w
    
    def fit(self, X_train, y_train, batch_size, lr, epochs=1):
        n_samples = X_train.shape[0]
        loss_history = []

        for epoch in range(epochs):

            # epoch_loss = 0.0
            n_batches  = 0

            for start in range(0, n_samples, batch_size):
                X_batch = X_train[start : start + batch_size]
                y_batch = y_train[start : start + batch_size]

                delta_w = self.backpropagation(X_batch, y_batch)

                for i, dw in enumerate(delta_w):
                    self.parameters[i] -= lr * dw

                y_pred, _, _ = self.forward_pass(X_batch)
                # epoch_loss += self._compute_loss(y_pred, y_batch)
                n_batches  += 1

            # epoch_loss /= n_batches
            # loss_history.append(epoch_loss)

        return loss_history
        
                
                
        
            
        
    



In [40]:
MLP = MultilayerPerceptron(sigmoid_activation, sigmoid_derivated, identity, [13, 50 ,1])

In [41]:
d = MLP.backpropagation(X_train, y_train)
d

C:\Users\sodre\AppData\Local\Temp\ipykernel_20568\386652744.py:2: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-z))


[array([[-6.65787301e-090,  0.00000000e+000,  4.75877601e-183,
         -4.42485350e-043,  3.77406453e-003,  0.00000000e+000,
         -1.48266973e+000,  0.00000000e+000,  7.93081928e-018,
          4.58421801e+000, -2.34085502e+000,  5.30935641e+000,
          0.00000000e+000,  0.00000000e+000, -5.59308774e-124,
         -3.55061125e+000,  1.66781943e+000,  3.19135266e+000,
          0.00000000e+000,  0.00000000e+000,  0.00000000e+000,
          1.68197025e-080, -1.06260065e+001, -1.75773614e-001,
         -2.77889276e-256, -1.82300316e+000, -7.39723378e+000,
          0.00000000e+000,  2.71197925e-001,  0.00000000e+000,
         -7.66599344e-146,  0.00000000e+000, -1.54893827e+000,
          4.56417415e-001,  7.90980539e-055, -2.91651310e-002,
          2.04969560e-001,  0.00000000e+000,  5.75926106e-206,
          4.33241615e-003,  0.00000000e+000, -8.03770175e-001,
          0.00000000e+000, -9.31132095e+000, -1.00458339e-141,
          1.26629211e+001,  4.00148085e+001,  8.5516223

In [56]:
MLP.fit(X_train, y_train, 5, 0.001, 50)

C:\Users\sodre\AppData\Local\Temp\ipykernel_20568\386652744.py:2: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-z))


[]

In [62]:
y_p, data, t = MLP.forward_pass(X_test)
y_p = y_p.flatten()
RMSE(y_test, y_p)

C:\Users\sodre\AppData\Local\Temp\ipykernel_20568\386652744.py:2: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-z))


np.float64(7.321706563902682)

## Part II 

For this exercise you will build and train a 2-layer neural network that predicts the exact digit from a hand-written image, using the MNIST dataset. 
For this exercise, add weight decay to your network.

In [63]:
from sklearn.datasets import load_digits

In [64]:
digits = load_digits()

In [65]:
X = digits.data
y = digits.target

In [66]:
X.shape

(1797, 64)

Again, you will split the data into training, validation and testing.

In [67]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size= 0.7)
X_test, X_val, y_test, y_val = train_test_split(X, y, train_size= 0.5)

In [69]:
MLP = MultilayerPerceptron(sigmoid_activation, sigmoid_derivated, softmax_activation, [64, 70, 10])

In [70]:
MLP.fit(X_train, y_train, 5, 0.0001, 30)

ValueError: cannot reshape array of size 5 into shape (10)